In [ ]:
import numpy as np
import pandas as pd
import duckdb as duckdb
import csv
import gzip

In [57]:
# don't use dtype=str as that creates legacy objects that pythons stores strings in
# pyarrow is a better option for string storage and manipulation (columnar storage)
df_title_basics = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz", 
    sep="\t", 
    quoting=csv.QUOTE_NONE,        # the fix: " is literal data, not a quote char
    #dtype_backend="pyarrow", # Create a lot of drama with nulls versus nan
)

In [33]:
with gzip.open("d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz", "rt", encoding="utf-8") as f:
    nlines = sum(1 for _ in f)     # generator: counts without holding the file in memory

print("lines:", nlines, "| records:", nlines - 1)   # minus header

lines: 12768517 | records: 12768516


In [34]:
df_title_basics.dtypes

tconst            string[pyarrow]
titleType         string[pyarrow]
primaryTitle      string[pyarrow]
originalTitle     string[pyarrow]
isAdult            int64[pyarrow]
startYear         string[pyarrow]
endYear           string[pyarrow]
runtimeMinutes    string[pyarrow]
genres            string[pyarrow]
dtype: object

In [36]:
len(df_title_basics)

12768516

In [37]:
# NB" Coerce makes non numeric nan and pyarrow sees nan as valid number - need to use numpy is nan instead of isna
nonnumberinminutes = np.isnan(pd.to_numeric(df_title_basics["runtimeMinutes"], errors='coerce'))

In [38]:
nonnumberinminutes.value_counts()

runtimeMinutes
True     8193746
False    4574770
Name: count, dtype: int64

In [39]:
showme = df_title_basics["runtimeMinutes"][nonnumberinminutes]

In [40]:
showme.value_counts()

runtimeMinutes
\N    8193746
Name: count, dtype: int64[pyarrow]

In [16]:
df_title_basics["rownum"] = range(len(df_title_basics))

In [17]:
df_title_basics.sample(n=10, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,rownum
2615756,tt13020394,tvEpisode,Episode #1.544,Episode #1.544,0,\N,\N,\N,Drama,2615756
8913370,tt37283892,tvSeries,Lost Boy,Lost Boy,0,\N,\N,\N,Thriller,8913370
11115785,tt6285120,tvEpisode,Episode #8.5,Episode #8.5,0,2016,\N,55,"Game-Show,Romance",11115785
5583584,tt21887000,tvEpisode,Episode #1.1808,Episode #1.1808,0,\N,\N,\N,\N,5583584
9032730,tt37782317,tvEpisode,08-18-2025,08-18-2025,0,2025,\N,\N,Talk-Show,9032730
9017446,tt37712410,tvEpisode,Episode #72.32,Episode #72.32,0,2025,\N,60,"News,Talk-Show",9017446
1156299,tt10342036,tvEpisode,Episode #1.138,Episode #1.138,0,2012,\N,\N,Drama,1156299
8077150,tt33348813,tvEpisode,22,22,0,2023,\N,\N,"Drama,Romance",8077150
10074131,tt43597228,tvEpisode,Ashes in the Ledger,Ashes in the Ledger,0,2026,\N,\N,Drama,10074131
1709366,tt11341544,tvEpisode,Episode #1.22,Episode #1.22,0,2014,\N,\N,Romance,1709366


In [24]:
df_title_basics[(df_title_basics["runtimeMinutes"] != r"\N") & (~df_title_basics["runtimeMinutes"].str.isdigit())].head(5)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,rownum
1096192,tt10233364,tvEpisode,Rolling in the Deep Dish Rolling in the Deep Dish,0,2019,\N,\N,Reality-TV,<NA>,1096192
1504608,tt10970874,tvEpisode,Die Bauhaus-Stadt Tel Aviv - Vorbild für die M...,0,2019,\N,45,Talk-Show,<NA>,1504608
1890737,tt11670006,tvEpisode,...ein angenehmer Unbequemer... ...ein angeneh...,0,1981,\N,30,Documentary,<NA>,1890737
2000902,tt11868642,tvEpisode,GGN Heavyweight Championship Lungs With Mike T...,0,2020,\N,\N,Talk-Show,<NA>,2000902
2153688,tt12149332,tvEpisode,Jeopardy! College Championship Semifinal Game ...,0,2020,\N,45,Game-Show,<NA>,2153688


In [21]:
dodgy_rows = df_title_basics[(df_title_basics["runtimeMinutes"] != r"\N") & (~df_title_basics["runtimeMinutes"].str.isdigit())]

In [23]:
dodgy_rows["rownum"].min()

np.int64(1096192)

In [30]:
import gzip

path = "d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz"
target = 1096192

with gzip.open(path, "rt", encoding="utf-8") as f:
    for lineno, line in enumerate(f, start=1):   # line 1 = header
        if lineno in range(target-2, target+3):
            print(lineno, repr(line))                    # repr: shows \t and stray quotes
            print(line.split("\t"))

1096190 'tt10233354\tmovie\tGoing Home for Tet\tVe Que an Tet\t0\t2018\t\\N\t80\tAdventure,Comedy,Family\n'
['tt10233354', 'movie', 'Going Home for Tet', 'Ve Que an Tet', '0', '2018', '\\N', '80', 'Adventure,Comedy,Family\n']
1096191 'tt10233356\ttvEpisode\tTriple Crown Pawn\tTriple Crown Pawn\t0\t2019\t\\N\t41\tReality-TV\n'
['tt10233356', 'tvEpisode', 'Triple Crown Pawn', 'Triple Crown Pawn', '0', '2019', '\\N', '41', 'Reality-TV\n']
1096192 "tt1023336\tmovie\tWhere's My Stuff?\tPortable Storage\t0\t2011\t\\N\t85\tComedy\n"
['tt1023336', 'movie', "Where's My Stuff?", 'Portable Storage', '0', '2011', '\\N', '85', 'Comedy\n']
1096193 'tt10233360\ttvEpisode\tLa venganza de Benjamín\tLa venganza de Benjamín\t0\t2019\t\\N\t\\N\tCrime,Drama,Thriller\n'
['tt10233360', 'tvEpisode', 'La venganza de Benjamín', 'La venganza de Benjamín', '0', '2019', '\\N', '\\N', 'Crime,Drama,Thriller\n']
1096194 'tt10233364\ttvEpisode\t"Rolling in the Deep Dish\t"Rolling in the Deep Dish\t0\t2019\t\\N\t\\N\tR

KeyboardInterrupt: 

In [45]:
df_title_basics["runtimeMinutes"] = pd.to_numeric(df_title_basics["runtimeMinutes"], errors='coerce')

In [47]:
df_title_basics["titleType"] = df_title_basics["titleType"].astype("category")

In [48]:
df_title_basics.dtypes

tconst            string[pyarrow]
titleType                category
primaryTitle      string[pyarrow]
originalTitle     string[pyarrow]
isAdult            int64[pyarrow]
startYear         string[pyarrow]
endYear           string[pyarrow]
runtimeMinutes    double[pyarrow]
genres            string[pyarrow]
dtype: object

In [ ]:
#convert nan to proper nulls

In [56]:
df_title_basics.groupby(["titleType"]).agg(avg_minutes=("runtimeMinutes", "mean"))

C:\Users\debee\AppData\Local\Temp\ipykernel_17320\3764259421.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_title_basics.groupby(["titleType"]).agg(avg_minutes=("runtimeMinutes", "mean"))


,avg_minutes
titleType,
movie,NaN
short,NaN
tvEpisode,NaN
tvMiniSeries,NaN
tvMovie,NaN
tvPilot,NaN
tvSeries,NaN
tvShort,NaN
tvSpecial,NaN
